In [21]:
from fastembed import LateInteractionTextEmbedding
from qdrant_client import QdrantClient, models
import time

### Embedding Model

In [3]:
colbert_model = LateInteractionTextEmbedding(
    model_name="colbert-ir/colbertv2.0"
)

In [4]:
colbert_model.embedding_size

128

### Simple Embedding

In [5]:
document = """This study examines the environmental benefits of 
electric bus fleets in three major metropolitan areas over a 
two-year period. Our analysis shows that electric buses reduce 
carbon emissions by an average of 65% compared to traditional 
diesel buses, while also decreasing noise pollution in urban 
centers by 40 decibels."""

In [6]:
document_embeddings = next(colbert_model.passage_embed([document]))
document_embeddings

array([[-0.03376379, -0.01941497,  0.19230309, ...,  0.02873551,
        -0.08662578, -0.1399344 ],
       [-0.11649974,  0.01376308,  0.177749  , ...,  0.01569705,
        -0.05300238, -0.11508605],
       [-0.10209138, -0.0631536 ,  0.09992975, ...,  0.03791303,
         0.07584608,  0.00922441],
       ...,
       [-0.09043387, -0.0612797 ,  0.10340663, ...,  0.07790045,
        -0.01314187,  0.09029499],
       [-0.13939631, -0.048699  ,  0.12846842, ...,  0.08749621,
        -0.03593391,  0.06377753],
       [-0.08001525, -0.05073933,  0.19033323, ...,  0.04507737,
        -0.02619245, -0.09820459]], shape=(55, 128), dtype=float32)

In [7]:
document_embeddings.shape

(55, 128)

In [8]:
query = "advantages of EV cars"

In [10]:
query_embeddings = next(colbert_model.query_embed([query]))

In [11]:
query_embeddings.shape

(32, 128)

### Qdrant

In [17]:
collection_name = "qdrant-tests-2"
colbert_vector_name = "colbert-ir/colbertv2.0"

In [15]:
client = QdrantClient("http://localhost:6333")

In [16]:
client.delete_collection(collection_name)

False

In [18]:
client.create_collection(
    collection_name,
    vectors_config={
        colbert_vector_name: models.VectorParams(
            size=colbert_model.embedding_size,
            distance=models.Distance.DOT,
            multivector_config=models.MultiVectorConfig(
                comparator=models.MultiVectorComparator.MAX_SIM,
            ),
            hnsw_config=models.HnswConfigDiff(m=0),
        ),
    },
)

True

In [19]:
documents = [
    "Qdrant is a vector database designed for similarity search applications",
    "SQL databases use structured tables with predefined schemas for data storage",
    "Using Qdrant you can store embeddings and perform efficient searches",
    "Traditional SQL queries filter data using exact matches and joins",
    "Qdrant supports multi-vector configurations for late interaction models like ColBERT",
    "SQL performs well for transactional workloads but lacks semantic search capabilities",
    "The Qdrant client allows you to create collections with custom distance metrics",
    "Migrating from SQL to vector databases enables similarity-based retrieval at scale",
    "Qdrant's MaxSim comparator enables token-level similarity scoring for multi-vectors",
    "SQL databases struggle with high-dimensional embeddings unlike specialized vector stores",
]

In [20]:
client.upsert(
    collection_name,
    points=[
        models.PointStruct(
            id=i,
            vector={
                colbert_vector_name: next(
                    colbert_model.passage_embed([document])
                ),
            },
            payload={"text": document},
        )
        for i, document in enumerate(documents, start=1)
    ],
)

UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

### Query

In [25]:
def colbert_query(q: str, limit: int = 5) -> list[dict]:
    start_time = time.monotonic()
    embedding = next(colbert_model.query_embed(q))
    end_time = time.monotonic()
    print("ColBERT vector generation time:", end_time - start_time)

    start_time = time.monotonic()
    result = client.query_points(
        collection_name,
        query=embedding,
        using=colbert_vector_name,
        limit=limit,
        with_payload=True,
    )
    end_time = time.monotonic()
    print("Query time:", end_time - start_time)

    return [point.payload for point in result.points]

In [26]:
query = "search performance in Qdrant"

In [27]:
colbert_hits = colbert_query(query)

ColBERT vector generation time: 0.04690800299977127
Query time: 0.10929776700049842


In [28]:
colbert_hits

[{'text': 'Using Qdrant you can store embeddings and perform efficient searches'},
 {'text': 'Qdrant is a vector database designed for similarity search applications'},
 {'text': "Qdrant's MaxSim comparator enables token-level similarity scoring for multi-vectors"},
 {'text': 'Qdrant supports multi-vector configurations for late interaction models like ColBERT'},
 {'text': 'The Qdrant client allows you to create collections with custom distance metrics'}]